# 数据清洗

**负责人：** 2 号　**阶段：** 1 数据准备

产出全组共用的清洗后数据，并记录每一步改动。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [ ] 和 4 号确认清洗数据的列和 `group_id` 的含义
- [x] 用 `ftfy` 修复编码；还原 HTML 实体（如 `&amp;`）
- [x] `__` 的来源不确定（可能是弯引号，也可能是 emoji）：先人工抽样核对，只还原核对过的模式（例如 `don__` → `don't`），其余统一替换成占位符；抽样核对可请 3 号协助
- [x] 去除完全重复；近似重复要么删除，要么分到同一个 `group_id`（必须在划分数据之前做）
- [x] 用 `langdetect` 识别语言，决定非英文样本怎么处理
- [x] hashtag 按驼峰拆词，`@xxx` 替换为 `@user`，删除清洗后为空的行
- [x] 生成 `text_full`（小写、去标点、词形还原、去停用词但保留 not / no / never）和 `text_light`（只修编码）
- [x] 保存前运行 `check_clean()`；每一步的规则、影响行数和前后示例写进 `docs/cleaning_log.md`

## 产出

- `DATA_ROOT/clean_v1/clean_v1.csv`，列为 `id, group_id, text_raw, text_full, text_light, label`
- `DATA_ROOT/clean_v1/cleaning_log.md`（`docs/cleaning_log.md` 保留副本）

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## 1. Raw data at a glance

One row is one tweet. The file has two columns, `text` and `text_sentiment`; `load_raw()` renames the label to `label` and adds a stable `id` (the row number in the raw file), which every later file uses to stay aligned.

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd

from src import cleaning
from src.config import CLEAN_DIR, CLEAN_VERSION, PROJECT_ROOT

pd.set_option("display.max_colwidth", 90)

raw = load_raw()
print(raw.shape)
raw["label"].map(LABELS).value_counts().to_frame("tweets")

## 2. Evidence for the `__` rule

In the source file the right single quote `’` **and the character after it** were replaced by two underscores: `it’s` → `it__`, `don’t` → `don__`, `you’re` → `you__e`. We printed every occurrence of each candidate pattern and checked them by hand before writing a restoration rule:

- negation stems (`don__`, `isn__`, `can__` …) → `n't`
- `I__` → `I'm`; `it__`, `that__`, `what__`, `there__` … → `'s`
- `you__e`, `we__e`, `they__e` → `'ve` when the next word (after adverbs such as *already*) is a past participle (*we've witnessed*), otherwise `'re` (*you're holding*)
- `i__l`, `you__l` … → `'ll`; `i__e` → `I've`; `y__ll` → `y'all`

Everything else cannot be restored reliably. A noun followed by `__` is a possessive about half of the time (`Biden__ speech`) and an emoji the other half (`Markets in deep Red__ #bearmarket`), so these become the placeholder `[UNK]`.

In [ ]:
def contexts(pattern, texts, width=22):
    """Every match of a compiled pattern with some surrounding text."""
    found = []
    for s in texts:
        for m in pattern.finditer(s):
            found.append(s[max(0, m.start() - width): m.end() + width].replace("\n", " "))
    return found


rules = [(name, pattern) for name, pattern, _ in cleaning.CONTRACTION_RULES]
rules.append(("'re/'ve", cleaning.RE_VE_RULE))
rows = []
for name, pattern in rules:
    hits = contexts(pattern, raw["text"])
    rows.append({"rule": name, "matches": len(hits), **{f"example {i + 1}": h for i, h in enumerate(hits[:3])}})
pd.DataFrame(rows)

In [ ]:
# '__' after a word that no rule covers: possessive 's or an emoji, which cannot be told apart
covered = set(cleaning.NEGATION_STEMS) | set(cleaning.S_STEMS) | {"i"}
other = re.compile(r"\b([A-Za-z]+)__(?=[^A-Za-z0-9_]|$)")
leftover = [s[max(0, m.start() - 22): m.end() + 22].replace("\n", " ")
            for s in raw["text"] for m in other.finditer(s) if m.group(1).lower() not in covered]
print(f"{len(leftover)} unresolved 'word__' occurrences, e.g.:")
for c in leftover[:10]:
    print("  ", c)

## 3. Run the cleaning pipeline

`cleaning.run_cleaning()` applies the steps in a fixed order and logs each one. The order matters:

1. exact duplicates are removed first, so they are not processed twice;
2. mentions are replaced **before** underscores are handled, because user names often contain underscores;
3. unreadable tweets are detected on the encoding-repaired text, before placeholders hide the residue;
4. language is detected **before** `#` is removed, so hashtag words such as `stockmarketcrash` do not confuse the detector;
5. content duplicates are found on the normalised `text_light`;
6. near-duplicate groups are built last, on the final rows.

In [ ]:
clean, log, dropped = cleaning.run_cleaning(raw)
log_df = pd.DataFrame(log)
log_df[["step", "operation", "rows_affected", "rows_remaining", "details"]]

In [ ]:
log_df[["operation", "example"]].query("example != ''")

## 4. Rows removed

No row was removed for being a minority opinion or an outlier in length: a 400-character tweet and a one-hashtag tweet are both legitimate tweets. Rows were removed only when they duplicate another row or cannot be read.

In [ ]:
print(dropped["reason"].value_counts().to_string())
dropped.groupby("reason").head(2)[["reason", "id", "label", "text_raw"]]

## 5. Language and labels

Readable non-English tweets (Spanish, French, Italian, romanised Hindi …) are **kept** and flagged in the `lang` column, so models can use them and a sensitivity analysis can drop them. `und` means the body is too short to tell (usually only hashtags and a few words).

The label mix differs sharply by language: non-English and undetermined tweets are mostly *Neutral*. This is what an English-only sentiment lexicon would produce, and supports the hypothesis that the labels were generated automatically (tested properly in `04_analysis/31_label_noise_errors.ipynb`).

In [ ]:
lang_group = clean["lang"].map(lambda l: "English" if l == "en" else ("Undetermined" if l == "und" else "Other languages"))
print(clean["lang"].value_counts().head(10).to_string())
share = pd.crosstab(lang_group, clean["label"].map(LABELS), normalize="index")
counts = lang_group.value_counts()
share[["Negative", "Neutral", "Positive"]].round(3)

In [ ]:
# Chart colours: sentiment is a polarity, so a diverging scheme (red - grey - blue) with direct labels
SURFACE, INK, INK_2, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#c3c2b7"
SENTIMENT_COLORS = {"Negative": "#e34948", "Neutral": "#c3c2b7", "Positive": "#2a78d6"}

order = ["English", "Other languages", "Undetermined"]
fig, ax = plt.subplots(figsize=(7, 2.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
left = pd.Series(0.0, index=order)
for label in ["Negative", "Neutral", "Positive"]:
    values = share.loc[order, label]
    ax.barh(order, values, left=left, height=0.55, color=SENTIMENT_COLORS[label],
            edgecolor=SURFACE, linewidth=2, label=label)
    for group, v, l in zip(order, values, left):
        if v >= 0.07:
            ax.text(l + v / 2, group, f"{v:.0%}", ha="center", va="center", fontsize=9, color=INK)
    left += values
ax.set_yticks(range(len(order)), [f"{g}  (n={counts[g]:,})" for g in order], color=INK_2, fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.tick_params(axis="x", colors=INK_2, labelsize=8)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(AXIS)
ax.set_title("Label share by detected language", loc="left", fontsize=11, color=INK, pad=24)
ax.legend(ncol=3, frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9, labelcolor=INK_2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "cleaning_label_by_language.png", dpi=200, facecolor=SURFACE)
plt.show()

## 6. Before and after

`text_light` keeps case, punctuation and the `[UNK]` placeholder for transformer models; `text_full` is the normalised, lemmatised version for the TF-IDF baseline and FastText.

In [ ]:
clean.sample(6, random_state=SEED)[["text_raw", "text_light", "text_full"]]

## 7. Cleaning funnel

In [ ]:
removed = dropped["reason"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(7, 2.4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.barh(removed.index, removed.values, height=0.55, color="#2a78d6")
for name, v in removed.items():
    ax.text(v + 0.4, name, str(v), va="center", fontsize=9, color=INK_2)
ax.tick_params(axis="y", colors=INK_2, labelsize=9)
ax.tick_params(axis="x", colors=INK_2, labelsize=8)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(AXIS)
ax.set_xlabel("tweets removed", color=INK_2, fontsize=9)
ax.set_title(f"Rows removed during cleaning ({len(raw):,} → {len(clean):,} tweets)", loc="left", fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "cleaning_funnel.png", dpi=200, facecolor=SURFACE)
plt.show()

## 8. Class balance after cleaning

Cleaning removed mostly *Neutral* rows (hashtag-only duplicates and unreadable tweets), so the imbalance between the largest and smallest class shrinks slightly. Stratified splitting, macro-F1 and class weights handle the rest.

In [ ]:
balance = pd.DataFrame({
    "before": raw["label"].map(LABELS).value_counts(),
    "after": clean["label"].map(LABELS).value_counts(),
})
balance["after share"] = (balance["after"] / balance["after"].sum()).round(3)
print("largest / smallest class:", round(balance["after"].max() / balance["after"].min(), 2))
balance

## 9. Save outputs

- `DATA_ROOT/clean_v{CLEAN_VERSION}/clean_v{CLEAN_VERSION}.csv`: the cleaned data every model reads (the versioned folder is tracked in Git)
- `DATA_ROOT/clean_v{CLEAN_VERSION}/dropped_v{CLEAN_VERSION}.csv`: removed rows with the reason, for auditing
- `DATA_ROOT/clean_v{CLEAN_VERSION}/cleaning_log.md`: the versioned cleaning log
- `docs/cleaning_log.md`: a copy of the latest cleaning log (report Table 4 and appendix)

In [ ]:
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(CLEAN_DIR / f"clean_v{CLEAN_VERSION}.csv", index=False, encoding="utf-8")
dropped.to_csv(CLEAN_DIR / f"dropped_v{CLEAN_VERSION}.csv", index=False, encoding="utf-8")

notes = [
    f"Data version: clean_v{CLEAN_VERSION}. Produced by notebooks/01_data/01_cleaning.ipynb with src/cleaning.py.",
    "[UNK] marks a character that cannot be recovered from the source file: an emoji, a symbol, or a possessive 's "
    "(the source replaced the apostrophe and the following character with '__'). In text_full it appears as the token 'unk'.",
    f"Readable non-English tweets are kept and flagged in the lang column "
    f"({int((clean['lang'] != 'en').sum())} rows, of which {int((clean['lang'] == 'und').sum())} are too short to tell).",
    "group_id links near-duplicate tweets; 02_splits must keep every group inside one fold.",
    "Extra columns: lang (detected language), n_restored (contractions restored), n_unk (characters replaced by [UNK]).",
]
log_text = cleaning.log_to_markdown(log, notes=notes)
log_path = CLEAN_DIR / "cleaning_log.md"
log_path.write_text(log_text, encoding="utf-8")
(PROJECT_ROOT / "docs" / "cleaning_log.md").write_text(log_text, encoding="utf-8")

reloaded = load_clean()  # re-reads the file and runs check_clean()
print(reloaded.shape, "| saved to", CLEAN_DIR)
reloaded.head(3)